In [3]:
# For API calls and response parsin
import pandas as pd
import requests
from pprint import pprint

# For connecting to Supabase
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

## Testing the API Connection

In [4]:
url = "https://clinicaltrials.gov/api/v2/studies"
params={"query.cond": "non-small cell lung cancer",
        "pageSize": 1000,
        "filter.advanced": "AREA[StartDate]2022"}
response = requests.get(url, params)
study = response.json()

## Inspecting the Return Object

In [5]:
for i in study["studies"]:
    print(i['protocolSection'])

{'identificationModule': {'nctId': 'NCT05230888', 'orgStudyIdInfo': {'id': 'V1.1 10.09.2021'}, 'organization': {'fullName': 'University of Hull', 'class': 'OTHER'}, 'briefTitle': 'Comprehensive Geriatric Assessment in Elderly Non-Small Cell Lung Cancer Patients', 'officialTitle': 'A Prospective Study of Comprehensive Geriatric Assessment (CGA) in Non-Small Cell Lung Cancer Patients (NSCLC) Undergoing Treatment With Immune Checkpoint Inhibitors.'}, 'statusModule': {'statusVerifiedDate': '2023-08', 'overallStatus': 'WITHDRAWN', 'whyStopped': 'Research personnel not recruited', 'expandedAccessInfo': {'hasExpandedAccess': False}, 'startDateStruct': {'date': '2022-02-09', 'type': 'ACTUAL'}, 'primaryCompletionDateStruct': {'date': '2022-11', 'type': 'ESTIMATED'}, 'completionDateStruct': {'date': '2022-11', 'type': 'ESTIMATED'}, 'studyFirstSubmitDate': '2022-01-12', 'studyFirstSubmitQcDate': '2022-01-27', 'studyFirstPostDateStruct': {'date': '2022-02-09', 'type': 'ACTUAL'}, 'lastUpdateSubmitD

In [6]:
pprint(study["studies"][2]['protocolSection'])

{'armsInterventionsModule': {'armGroups': [{'description': 'BLU-451 '
                                                           'monotherapy with '
                                                           'dose escalation in '
                                                           'participants with '
                                                           'metastatic cancer '
                                                           'with EGFR Ex20ins '
                                                           'or other selected '
                                                           'EGFR mutations '
                                                           'that have '
                                                           'progressed after '
                                                           'prior systemic '
                                                           'therapies.',
                                            'interventionNames': ['Drug:

## Identifying Key Variables in the Response Object

#### Study Identifiers

- NCT ID: study['studies'][2]['protocolSection']['identificationModule']['nctId']
- Sponsor's internal trial ID: study["studies"][2]['protocolSection']['identificationModule']['orgStudyIdInfo']['id']
- Study Acronym: study['studies'][2]['protocolSection']['identificationModule']['acronym']
- Study Title: study['studies'][2]['protocolSection']['identificationModule']['briefTitle']
- Long Title: study['studies'][2]['protocolSection']['identificationModule']['officialTitle']

#### Sponsor Information

- Lead sponsor: study['studies'][2]['protocolSection']['sponsorCollaboratorsModule']['leadSponsor']
- Collaborators: study['studies'][2]['protocolSection']['sponsorCollaboratorsModule']['collaborators']

#### Study Design

- Arms: study['studies'][2]['protocolSection']['armsInterventionsModule']['armGroups']
- Drugs: study['studies'][2]['protocolSection']['armsInterventionsModule']['interventions']
- Conditions: study['studies'][2]['protocolSection']['conditionsModule']['conditions']
- Endpoints: study['studies'][2]['protocolSection']['outcomesModule']
- Line of Therapy: study['studies'][2]['protocolSection']['descriptionModule']['briefSummary'] --> can be extracted from this text
- Main investigational Drug; study['studies'][2]['protocolSection']['descriptionModule']['briefSummary'] --> can be extracted from this text

#### Study Timeline

- Study status: study['studies'][2]['protocolSection']['statusModule']['overallStatus']
- Start date: study['studies'][2]['protocolSection']['statusModule']

In [7]:
# NCT Number
for s in study['studies'][2]['derivedSection']:
    print(s)

miscInfoModule
conditionBrowseModule
interventionBrowseModule


# Response Parser

Workshopping the response parser class to extrac the key datafields from the JSON response object. Rough sketch:

- Process each section individually
- Include error handling / validation logic
- Package results in a pandas dataframe

Next Columns to Debug:

- Arms
- Collaborators

In [26]:
# copying the response object to change variable names
api_response_json = study.copy()

# Flattener function
def flatten_study(study: dict) -> dict:
    '''Flattens the study dictionary of the JSON API response into a single dictionary to turn into a DataFrame'''
    id_mod = study['protocolSection'].get("identificationModule", {})
    sponsor_mod = study['protocolSection'].get("sponsorCollaboratorsModule", {})
    arm_mod = study['protocolSection'].get("armsInterventionsModule", {})
    cond_mod = study['protocolSection'].get("conditionsModule", {})
    outcome_mod = study['protocolSection'].get("outcomesModule", {})
    desc_mod = study['protocolSection'].get("descriptionModule", {})
    status_mod = study['protocolSection'].get("statusModule", {})

    extracted_dict = {
        "nct" : id_mod.get("nctId"),
        "study_acronym": id_mod.get("acronym"),
        "study_title": id_mod.get("briefTitle"),
        "study_long_title": id_mod.get("officialTitle"),
        "lead_sponsor": sponsor_mod.get("leadSponsor", {}).get("name"),
        "conditions": cond_mod.get("conditions"),
        "endpoints": outcome_mod.get("outcomesModule"),
        "status": status_mod.get("overallStatus"),
        "start_date": status_mod.get("statusModule")
    }
    
    return extracted_dict

# Prelim testing
results = pd.DataFrame.from_dict([flatten_study(s) for s in api_response_json['studies']])

# Connecting to Supabase

In [19]:
load_dotenv()

True

In [20]:
client = create_engine(os.getenv("SUPABASE_PROJECT_URL"))

In [28]:
results.to_sql(name="studies",
               con=client,
               if_exists="replace",
               index=False)

-1